DATA AUDIT


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3, os

df = pd.read_csv("data/capstone_safety.csv")

print("=== SHAPE ==="); print(df.shape)
print("\n=== DTYPES ==="); print(df.dtypes)
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")
print("\n=== NUMERIC SUMMARY ===")
print(df.describe())



=== SHAPE ===
(250, 15)

=== DTYPES ===
project_id                   str
city                         str
zone                         str
sector                       str
contractor                   str
project_value_cad        float64
workers_on_site          float64
safety_incidents         float64
near_misses              float64
lost_time_incidents        int64
delay_days                 int64
project_duration_days      int64
approved                     str
project_type                 str
season_started               str
dtype: object

=== MISSING VALUES ===
project_id                0
city                      0
zone                      0
sector                    0
contractor                0
project_value_cad         0
workers_on_site          17
safety_incidents         26
near_misses               9
lost_time_incidents       0
delay_days                0
project_duration_days     0
approved                  0
project_type              0
season_started            0
dtype: 

DATA CLEANING

In [2]:

df = df.drop_duplicates().reset_index(drop=True)

df = df[df["safety_incidents"].notna()]

df["workers_on_site"] = df["workers_on_site"].fillna(df["workers_on_site"].median())

df["near_misses"] = df["near_misses"].fillna(df["near_misses"].median())

# new derived column: incident_rate
# incident_rate = safety_incidents per 100 workers
# This normalises incident counts by site size — essential for fair comparison
# A site with 200 workers and 10 incidents is SAFER than one with
# 20 workers and 10 incidents. Raw counts are misleading.
df["incident_rate"] = (df["safety_incidents"] / df["workers_on_site"]) * 100
df["incident_rate"] = df["incident_rate"].round(2)

# derived column: near_miss_rate
# Same formula as incident_rate but using near_misses
# near_miss_rate = (near_misses / workers_on_site) * 100
df["near_miss_rate"] = (df["near_misses"] / df["workers_on_site"]) * 100
df["near_miss_rate"] = df["near_miss_rate"].round(2)

# Final audit
print(f"Final shape: {df.shape}")
print(f"Remaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

os.makedirs("data", exist_ok=True)
df.to_csv("data/capstone_safety_clean.csv", index=False)
print("✓ Clean data saved")

Final shape: (224, 17)
Remaining nulls:
Series([], dtype: int64)
✓ Clean data saved


SQL QUERIES

In [3]:
sectors_df = df[["sector"]].drop_duplicates().reset_index(drop=True)
sectors_df.insert(0, "sector_id", range(1, len(sectors_df)+1))

regions_df = df[["city","zone"]].drop_duplicates().reset_index(drop=True)
regions_df.insert(0, "region_id", range(1, len(regions_df)+1))

# Merge IDs back into projects
projects_db = df.merge(sectors_df, on="sector", how="left")
projects_db = projects_db.merge(regions_df, on=["city","zone"], how="left")

# Build database
conn = sqlite3.connect("data/capstone_safety.db")
sectors_df.to_sql("sectors",  conn, if_exists="replace", index=False)
regions_df.to_sql("regions",  conn, if_exists="replace", index=False)
projects_db.to_sql("projects", conn, if_exists="replace", index=False)

# --- Query 1: Safety incidents by sector ---
query1 = """
SELECT 
    sector,
    COUNT(*) AS project_count,
    AVG(incident_rate) AS avg_incident_rate,
    SUM(safety_incidents) AS total_incidents
FROM projects
GROUP BY sector
ORDER BY avg_incident_rate DESC;
"""
pd.read_sql_query(query1, conn)



# --- Query 2: Safety incidents by zone (Northern/Central/Southern) ---
query2 = """
SELECT 
    zone,
    COUNT(*) AS project_count,
    AVG(incident_rate) AS avg_incident_rate,
    SUM(safety_incidents) AS total_incidents
FROM projects
GROUP BY zone
ORDER BY avg_incident_rate DESC;
"""
pd.read_sql_query(query2, conn)


# --- Query 3: Incident rate by project_type ---
query3 = """
SELECT 
    project_type,
    COUNT(*) AS project_count,
    AVG(incident_rate) AS avg_incident_rate,
    SUM(safety_incidents) AS total_incidents
FROM projects
GROUP BY project_type
ORDER BY avg_incident_rate DESC;
"""
pd.read_sql_query(query3, conn)


# --- Query 4: Does season affect safety outcomes? ---
query4 = """
SELECT 
    season_started,
    COUNT(*) AS project_count,
    AVG(incident_rate) AS avg_incident_rate,
    SUM(safety_incidents) AS total_incidents
FROM projects
GROUP BY season_started
ORDER BY avg_incident_rate DESC;
"""
pd.read_sql_query(query4, conn)


conn.close()

VISUALIZATION

In [4]:

os.makedirs("charts", exist_ok=True)
sns.set_style("darkgrid")
plt.rcParams["figure.dpi"] = 120

# --- Chart 1 ---
# QUESTION: Which sector shows the highest average incident rate?

plt.figure(figsize=(8,5))
sector_rates = df.groupby("sector")["incident_rate"].mean().sort_values(ascending=False)

sns.barplot(x=sector_rates.values, y=sector_rates.index, palette="viridis")
plt.title("Oil & Gas Projects Show the Highest Average Incident Rates")
plt.xlabel("Average Incident Rate (per 100 workers)")
plt.ylabel("Sector")

plt.tight_layout()
plt.savefig("charts/chart_01_incident_rate_by_sector.png")
plt.close()

# --- Chart 2 ---
# QUESTION: Does Northern Alberta have higher incident rates than Central or Southern zones?

plt.figure(figsize=(7,5))
sns.boxplot(data=df, x="zone", y="incident_rate", palette="magma")
plt.title("Northern Zone Shows Higher Incident Rate Variability")
plt.xlabel("Zone")
plt.ylabel("Incident Rate (per 100 workers)")

plt.tight_layout()
plt.savefig("charts/chart_02_incident_rate_by_zone.png")
plt.close()

# --- Chart 3 ---
# QUESTION: Do larger construction sites experience more safety incidents?

plt.figure(figsize=(7,5))
sns.scatterplot(
    data=df,
    x="workers_on_site",
    y="safety_incidents",
    hue="sector",
    palette="tab10",
    alpha=0.7
)
plt.title("Larger Sites Tend to Record More Safety Incidents")
plt.xlabel("Workers on Site")
plt.ylabel("Safety Incidents")

plt.tight_layout()
plt.savefig("charts/chart_03_workers_vs_incidents.png")
plt.close()

# --- Chart 4 ---
# QUESTION: Are winter projects associated with higher incident rates?

plt.figure(figsize=(7,5))
season_rates = df.groupby("season_started")["incident_rate"].mean().sort_values(ascending=False)

sns.barplot(x=season_rates.index, y=season_rates.values, palette="coolwarm")
plt.title("Winter Projects Show the Highest Average Incident Rates")
plt.xlabel("Season Started")
plt.ylabel("Average Incident Rate (per 100 workers)")

plt.tight_layout()
plt.savefig("charts/chart_04_incident_rate_by_season.png")
plt.close()

# --- Chart 5 (optional but recommended) ---
# QUESTION: Which sector records the highest near-miss rate?

plt.figure(figsize=(8,5))
nm_rates = df.groupby("sector")["near_miss_rate"].mean().sort_values(ascending=False)

sns.barplot(x=nm_rates.values, y=nm_rates.index, palette="crest")
plt.title("Industrial and Oil & Gas Sectors Show Elevated Near-Miss Rates")
plt.xlabel("Average Near-Miss Rate (per 100 workers)")
plt.ylabel("Sector")

plt.tight_layout()
plt.savefig("charts/chart_05_near_miss_rate_by_sector.png")
plt.close()

print("✓ All charts saved")

C:\Users\nidhi\AppData\Local\Temp\ipykernel_32192\2895844204.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=sector_rates.values, y=sector_rates.index, palette="viridis")
C:\Users\nidhi\AppData\Local\Temp\ipykernel_32192\2895844204.py:24: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="zone", y="incident_rate", palette="magma")
C:\Users\nidhi\AppData\Local\Temp\ipykernel_32192\2895844204.py:59: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=season_rates.index, y=season_rates.values, palette="coolwarm")
C:\Users\nidhi\AppDat

✓ All charts saved
